In [1]:
import torch
from setting import BASE_DIR, DEVICE
import os
from cnn_lstm import ConvNet

model_path = os.path.join(BASE_DIR, 'trained_model', 'cnn-lstm', 'best_model.pth')
model = ConvNet()
checkpoint = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()


Data folder: /Users/nguyenhuynh/Documents/heart_pressure/ppg_split_files
Number of training files: 64
Number of validation files: 14
Number of test files: 14


ConvNet(
  (layer1): Sequential(
    (0): Conv1d(1, 128, kernel_size=(9,), stride=(1,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv1d(128, 256, kernel_size=(9,), stride=(1,))
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  )
  (adaptive): AdaptiveMaxPool1d(output_size=4)
  (lstm): LSTM(256, 56, batch_first=True)
  (fc1): Linear(in_features=224, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=1, bias=True)
)

In [3]:
from torch.utils.data import DataLoader
from dataset import PPGDataset, test_files
import torch
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Chuẩn bị DataLoader test (như bạn đã có)
test_dataset_sbp = PPGDataset(test_files, label='SBP')
print(f"Test dataset size: {len(test_dataset_sbp)}")
test_dataloader = DataLoader(test_dataset_sbp, batch_size=1024, shuffle=False)

# Biến lưu kết quả
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in tqdm(test_dataloader, desc="Validating", leave=False):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE).view(-1, 1)
        outputs = model(inputs)

        all_preds.extend(outputs.cpu().numpy().flatten())
        all_targets.extend(targets.cpu().numpy().flatten())

# Chuyển sang numpy array
all_preds = np.array(all_preds)
print(f"Predictions shape: {all_preds.shape}")
all_targets = np.array(all_targets)
print(f"Targets shape: {all_targets.shape}")

# Tính toán các metric
mae = mean_absolute_error(all_targets, all_preds)
mse = mean_squared_error(all_targets, all_preds)
rmse = np.sqrt(mse)
r2 = r2_score(all_targets, all_preds)

print(f"MAE  : {mae:.4f} mmHg")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f} mmHg")
print(f"R²   : {r2:.4f}")


Test dataset size: 1360000


KeyboardInterrupt: 